# 07.8 Concatenation Performance and the `textwrap` Module

Two practical topics: how to build strings **efficiently**, and the standard
library module that handles paragraph formatting so you do not have to.

## Easy — Building strings the right way

One rule: collect the parts, then join once.

In [ ]:
# EXAMPLE 1: Joining a list of words
words = ["Python", "is", "readable"]

print(" ".join(words))

In [ ]:
# EXAMPLE 2: Joining with different separators
words = ["a", "b", "c"]

print(" ".join(words))
print(", ".join(words))
print(" - ".join(words))
print("".join(words))

In [ ]:
# EXAMPLE 3: join() needs strings
# Numbers must be converted first.
numbers = [1, 2, 3]

try:
    ", ".join(numbers)
except TypeError as error:
    print("TypeError:", error)

print("Fixed:", ", ".join(str(number) for number in numbers))

In [ ]:
# EXAMPLE 4: Collecting parts in a loop, then joining
# The standard pattern for building text in a loop.
parts = []

for number in range(5):
    parts.append(f"item {number}")

print("\n".join(parts))

## Medium — Why join beats repeated concatenation

Strings are immutable, so every + builds a whole new string.

In [ ]:
# EXAMPLE 5: Timing the two approaches
import time


def with_plus(count):
    """Build by repeated concatenation."""
    result = ""
    for number in range(count):
        result += str(number)
    return result


def with_join(count):
    """Collect parts, join once."""
    return "".join(str(number) for number in range(count))


size = 30000

start = time.perf_counter()
with_plus(size)
plus_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
with_join(size)
join_ms = (time.perf_counter() - start) * 1000

print(f"repeated +=: {plus_ms:8.1f} ms")
print(f"''.join():   {join_ms:8.1f} ms")
print(f"join was about {plus_ms / join_ms:.0f}x faster")

In [ ]:
# EXAMPLE 6: Why it happens
# Each += copies everything built so far, so the work grows quadratically.
print("Building 'abcd' with += does this:")
print("   ''    + 'a'  -> copy 0 characters")
print("   'a'   + 'b'  -> copy 1 character")
print("   'ab'  + 'c'  -> copy 2 characters")
print("   'abc' + 'd'  -> copy 3 characters")
print("")
print("Total copies for n items: 0+1+2+...+(n-1), which is O(n squared).")
print("join() measures the total size once, allocates once, and fills it.")

In [ ]:
# EXAMPLE 7: Comparing all the building methods
import time
import io

count = 20000


def time_it(operation):
    """Return milliseconds for one call."""
    start = time.perf_counter()
    operation()
    return (time.perf_counter() - start) * 1000


def by_plus():
    result = ""
    for number in range(count):
        result += str(number)
    return result


def by_list_join():
    parts = []
    for number in range(count):
        parts.append(str(number))
    return "".join(parts)


def by_generator_join():
    return "".join(str(number) for number in range(count))


def by_stringio():
    buffer = io.StringIO()
    for number in range(count):
        buffer.write(str(number))
    return buffer.getvalue()


methods = [
    ("repeated +=", by_plus),
    ("list then join", by_list_join),
    ("generator join", by_generator_join),
    ("io.StringIO", by_stringio),
]

for label, method in methods:
    print(f"{label:<18} {time_it(method):8.1f} ms")

In [ ]:
# EXAMPLE 8: When += is perfectly fine
# For a handful of pieces, readability wins - the cost is irrelevant.
first = "Hello"
second = "World"

message = first + " " + second
print(message)
print("")
print("The rule only matters inside loops with many iterations.")

## Hard — textwrap and the string module

Formatting help you would otherwise write yourself.

In [ ]:
# EXAMPLE 9: Wrapping a long paragraph
import textwrap

paragraph = ("Python is a high-level, general-purpose programming language "
             "that emphasises code readability with its notable use of "
             "significant indentation.")

print(textwrap.fill(paragraph, width=50))

In [ ]:
# EXAMPLE 10: Getting a list of lines instead
import textwrap

paragraph = "The quick brown fox jumps over the lazy dog near the river bank"

lines = textwrap.wrap(paragraph, width=25)

for number, line in enumerate(lines, start=1):
    print(f"{number}: {line}")

In [ ]:
# EXAMPLE 11: Indenting wrapped text
import textwrap

paragraph = ("This paragraph has a hanging indent, which is useful for "
             "bullet points and numbered lists in terminal output.")

print(textwrap.fill(paragraph, width=50,
                    initial_indent="* ",
                    subsequent_indent="  "))

In [ ]:
# EXAMPLE 12: Removing common leading whitespace
import textwrap

# A triple-quoted string inside a function is indented with the code.
def get_help():
    """Return help text without the code indentation."""
    raw = """
        Usage: tool [options]

        Options:
          -h   show help
    """
    return textwrap.dedent(raw).strip()


print(get_help())

In [ ]:
# EXAMPLE 13: Shortening text with an ellipsis
import textwrap

title = "A Very Long Article Title That Will Not Fit In The Column"

print(textwrap.shorten(title, width=30))
print(textwrap.shorten(title, width=30, placeholder=" ..."))

In [ ]:
# EXAMPLE 14: Indenting every line
import textwrap

message = "line one\nline two\nline three"

print(textwrap.indent(message, "    "))

In [ ]:
# EXAMPLE 15: Indenting only some lines
import textwrap

message = "keep this\nindent this\nkeep this too"

# The predicate decides which lines get the prefix.
result = textwrap.indent(
    message,
    "> ",
    predicate=lambda line: "indent" in line,
)

print(result)

In [ ]:
# EXAMPLE 16: The string module constants
import string

print("ascii_lowercase:", string.ascii_lowercase)
print("ascii_uppercase:", string.ascii_uppercase)
print("digits:         ", string.digits)
print("punctuation:    ", string.punctuation)
print("whitespace:     ", repr(string.whitespace))

In [ ]:
# EXAMPLE 17: Using the constants for validation
import string


def describe(text):
    """Report which character classes appear in the text."""
    return {
        "letters": any(c in string.ascii_letters for c in text),
        "digits": any(c in string.digits for c in text),
        "punctuation": any(c in string.punctuation for c in text),
    }


for candidate in ["hello", "hello123", "hello!"]:
    print(f"{candidate:<10} {describe(candidate)}")

In [ ]:
# EXAMPLE 18: string.Template for safe substitution
# Template does not evaluate expressions, so it is safe for user-supplied text.
from string import Template

template = Template("Hello $name, you have $count messages")

print(template.substitute(name="Asha", count=3))

In [ ]:
# EXAMPLE 19: Why Template is safer than f-strings for untrusted templates
from string import Template

# safe_substitute leaves unknown placeholders alone rather than raising.
template = Template("Hello $name, your code is $code")

print(template.safe_substitute(name="Asha"))
print("")
print("Template cannot execute code. An f-string built from user input")
print("could evaluate arbitrary expressions, which is a security risk.")

## Takeaways

1. Build strings with **`"".join(parts)`**, not repeated `+=` in a loop.
2. `+=` is **O(n²)** because each step copies everything so far; `join` measures
   once and allocates once.
3. For a handful of pieces, `+` is perfectly fine — the rule is about loops.
4. `io.StringIO` is a good alternative when you are writing incrementally.
5. **`textwrap.fill()`** wraps a paragraph, **`wrap()`** returns lines,
   **`dedent()`** removes code indentation, **`shorten()`** truncates neatly.
6. `string.ascii_letters`, `digits` and `punctuation` save you typing character
   sets.
7. **`string.Template`** is the safe choice for templates that come from users —
   it cannot execute code.

## Try it yourself

1. Build a 50,000-character string both ways and time them.
2. Wrap a long paragraph to 40 columns with a hanging indent.
3. Use `dedent()` on a triple-quoted string inside a function.
4. Shorten a long title to 25 characters with a custom placeholder.
5. Validate that a password contains letters, digits and punctuation.